In [ ]:
#!pip uninstall -y transformers tokenizers accelerate peft triton -q

!pip install -q \
  transformers==4.38.2 \
  tokenizers==0.15.2 \
  accelerate==0.27.2 \
  peft==0.8.2 \
  sentencepiece \
  einops

# uninstalling triton after is necessary cuz of versin bugs
!pip uninstall -y triton -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:

%cd /content/drive/MyDrive/DSCI-691-Project/

/content/drive/.shortcut-targets-by-id/178re1vXBOn1poHWHRN9tiWVeMNB2tYhp/DSCI-691-Project


In [ ]:
!rm -rf ~/.cache/huggingface

from transformers import AutoTokenizer, AutoModel
from transformers.models.bert.configuration_bert import BertConfig

model_name = "zhihan1996/DNABERT-2-117M"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

config = BertConfig.from_pretrained(model_name)
config.pad_token_id = tokenizer.pad_token_id

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    config=config
)

import torch
from transformers import AutoTokenizer, BertConfig, BertModel


MODEL_NAME = "zhihan1996/DNABERT-2-117M"

tokenizer = AutoTokenizer.from_pretrainedm
config = BertConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = 0
model = BertModel.from_pretrained("zhihan1996/DNABERT-2-117M", config=config)


dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
hidden_states = model(inputs)[0]

embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape)

embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape)

# 1. Model and Function definitions

## 1.1 Dataset stuff

In [ ]:
from datasets import load_dataset
from typing import Optional
import pandas as pd

# load data

# temp dataset for diagnostic
#dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
#promoter_dataset = dataset.filter(lambda example: example["task"] == "promoter_all")
#print(promoter_dataset)


#sequences = dataset["train"]["sequence"]
#labels = dataset["train"]["label"]

# custom csv data


def load_data(paths: Optional[dict]=None):
  """
  Im makint this a sort of dict based thing, just so I can load a bunch of things into
  one ds with one call. If unspecified it loads a default dataset from huggingface that
  is very similar but not exactly like what we are doing. Mostly for easy diagnostic stuff
  """
  ds = {}
  if paths is None:
    # load temp dataset for diagnostic
    dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
    promoter_dataset = dataset.filter(lambda example: example["task"] == "promoter_all")

    train_df = pd.DataFrame({"sequence": promoter_dataset["train"]["sequence"],
                             "label": promoter_dataset["train"]["label"]})

    test_df = pd.DataFrame({"sequence": promoter_dataset["test"]["sequence"],
                             "label": promoter_dataset["test"]["label"]})

    val_df = train_df.sample(int(len(train_df) * 0.1), random_state=42) #0.1 val split from training
    train_df = train_df.drop(val_df.index)

    ds = {"train": train_df, "val": val_df, "test": test_df}
  else:
    # load custom csv data
    for fold, path in paths.items():
      df = pd.read_csv(path)
      ds[fold] = df[["sequence","label"]]

  return ds

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class DNAdataset(Dataset):
  def __init__(self, ds, tokenizer, max_length=128):
    self.tokenizer = tokenizer
    self.max_length = max_length
    self.sequences = ds['sequence'].reset_index(drop=True)
    self.labels = torch.tensor(ds['label'].values, dtype=torch.long)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    enc = self.tokenizer(
        self.sequences.iloc[idx],
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )
    out = {
        "input_ids": enc["input_ids"].squeeze(0),
        "attention_mask": enc["attention_mask"].squeeze(0),
        "label": self.labels[idx]
        }
    return out

### test dataloader
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
ds = load_data()

train = DNAdataset(ds['train'], tokenizer)
train_loader = DataLoader(train, batch_size=16, shuffle=True)

batch = next(iter(train_loader))
print(batch["input_id"].shape, batch["attn_mask"].shape, batch["label"].shape)

## 1.2 model defs

In [ ]:
# model building now on top of DNABERT2 hf model
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel, AutoTokenizer, BertConfig
from peft import get_peft_model, LoraConfig, TaskType

class DNABERT2Classifier(nn.Module):
  def __init__(self, checkpoint=None, pad_token=0, num_labels=2, dropout_prob=0.2,
               use_lora=True, lora_r=8, lora_alpha=16, lora_dropout=0.1):
    super().__init__()

    # official DNABERT2 model uses a depreciated version of triton flash attention
    self.MODEL_NAME = "zhihan1996/DNABERT-2-117M" #"quietflamingo/dnabert2-no-flashattention" or "zhihan1996/DNABERT-2-117M"
    self.num_labels = num_labels
    self.config = BertConfig.from_pretrained(self.MODEL_NAME)
    self.config.pad_token_id = pad_token
    base = AutoModel.from_pretrained(
        checkpoint if checkpoint else self.MODEL_NAME,
        trust_remote_code=True,
        config=self.config
    )

    if use_lora:
      lora_config = LoraConfig(
          r=lora_r,
          lora_alpha=lora_alpha,
          lora_dropout=lora_dropout,
          bias="none",
          target_modules=["Wqkv"],
      )
      self.encoder = get_peft_model(base, lora_config)
    else:
        self.encoder = base

    # class head
    hidden_size = self.config.hidden_size
    self.classifier = nn.Sequential(
      nn.LayerNorm(hidden_size),
      nn.Dropout(dropout_prob),
      nn.Linear(hidden_size,hidden_size//2),
      nn.GELU(),
      nn.Dropout(dropout_prob),
      nn.Linear(hidden_size//2,num_labels)
    )



  def forward(self, input_ids, attn_mask):
    outputs = self.encoder(input_ids, attention_mask=attn_mask)
    hidden = outputs[0]
    # cls token based
    # cls_rep = outputs.last_hidden_state[:,0,:]
    # pooled = hidden[:, 0, :]

    # mean pooling based
    mask_expanded = attn_mask.unsqueeze(-1).float()
    sum_hidden = (hidden*mask_expanded).sum(dim=1)
    sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
    pooled = sum_hidden/sum_mask

    logits = self.classifier(pooled)

    return logits


### test
dna = "ATCGATCGATCGATCGACATCGATCGAT"

inputs = tokenizer(dna, return_tensors='pt')
classifier = DNABERT2Classifier()
with torch.no_grad():
  logits = classifier(inputs["input_ids"], inputs["attention_mask"])

print(logits.shape)

## 1.3 training loop

In [ ]:
from transformers import DataCollatorWithPadding, get_linear_schedule_with_warmup

# training loop
def train_model(model, tokenizer, train, val, lr=1e-5, batch_size=16,
                epochs=10, patience=5, class_weights=None, freeze_base=False,
                save_path="best_model.pth"):
  torch.manual_seed(42)
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)

  # frozen encoder params
  if freeze_base:
    for p in model.encoder.parameters():
      p.requires_grad = False

  opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
  loss_fn = nn.CrossEntropyLoss(weight=class_weights)

  # new - collator
  collator = DataCollatorWithPadding(
      tokenizer=tokenizer,
      padding=True,
      return_tensors="pt"
      )
  train_ds = DataLoader(
      train,
      batch_size=batch_size,
      shuffle=True,
      collate_fn=collator
      )
  val_ds = DataLoader(
      val,
      batch_size=batch_size,
      shuffle=False,
      collate_fn=collator
      )


  # new scheduer
  total_steps = epochs * len(train_ds)

  scheduler = get_linear_schedule_with_warmup(
      opt,
      num_warmup_steps=int(0.1 * total_steps),
      num_training_steps=total_steps
  )

  epoch = 0
  no_imp = 0
  best_val_loss = None

  #while epoch < epochs and no_imp < patience:

  epoch = 0
  no_imp = 0
  best_val_loss = float("inf")

  for epoch in range(epochs):
    print(f'{'=' * 25} Epoch: {epoch} {'=' * 25}')

    train_loss = 0
    model.train()
    for batch in tqdm(train_ds, leave=False):
      batch = {k: v.to(device) for k, v in batch.items()}
      opt.zero_grad()
      logits = model(batch["input_ids"], batch["attention_mask"])
      loss = loss_fn(logits, batch["labels"])
      loss.backward()
      torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )
      opt.step()
      scheduler.step()
      train_loss += loss.item()

    train_loss /= len(train_ds)
    print(f'Train loss: {train_loss}')

    val_loss = 0
    model.eval()
    for batch in tqdm(val_ds, leave=False):
      batch = {k: v.to(device) for k, v in batch.items()}
      with torch.no_grad():
        logits = model(batch["input_ids"], batch["attention_mask"])
        loss = loss_fn(logits, batch["labels"])
        val_loss += loss.item()

    val_loss /= len(val_ds)
    print(f'Val loss: {val_loss}')

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      no_imp = 0
      torch.save(model.state_dict(), save_path)
    else:
      no_imp += 1

    print(f'best val loss: {best_val_loss:.2f} no imp epochs: {no_imp}')
    if no_imp >= patience:
        print("Early stopping triggered.")
        break



  model.load_state_dict(
      torch.load(save_path)
  )
  return model





### small test
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

ds = load_data()
train = DNAdataset(ds['train'], tokenizer)
val = DNAdataset(ds['val'], tokenizer)

classifier = DNABERT2Classifier()

train_model(classifier, train, val)

## 1.4 Eval stuff

In [ ]:
def predict(model, dataset, tokenizer, batch_size=16):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt"
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator
        )

    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad():
      for batch in tqdm(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"])
        prob = torch.softmax(logits, dim=1)[:, 1]
        ps.extend(logits.argmax(1).cpu().numpy())
        probs.extend(prob.cpu().numpy())
        ys.extend(batch["labels"].cpu().numpy())
    return ys, ps, probs

ds = load_data()
test_ds = DNAdataset(ds['test'], tokenizer)
ys, ps, probs = predict(model, test_ds)
record('test', 'test', ys, ps, probs)

## 1.5 Utils

In [ ]:
def record(model_name, split_label, y_true, y_pred, y_prob=None):
    row = {
        'Model': model_name,
        'Split': split_label,
        'f1': f1_score(y_true, y_pred, average='macro'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
    }
    print(f"  {model_name:<16} {split_label:<10} "
          f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}  "
          f"AUC={row['roc_auc']:.4f}" if y_prob is not None
          else f"  {model_name:<16} {split_label:<10} "
               f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}")
    return row

# 2 experimental result -  dataset G

## 2.1 load data

In [ ]:
# imports
from datasets import load_dataset
from typing import Optional
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# load data
paths_dict = {
    'train': 'dataset_g/train.csv',
    'test-heldout': 'dataset_g/test_OOD.csv',
    'test-id': 'dataset_g/test_ID.csv',
    'test-matched_id': 'dataset_g/test_motif_matched_ID.csv'
    }

ds = load_data(paths_dict)
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# dataset - heldout
train_df, val_df = train_test_split(ds['train'], test_size=0.2, random_state=42)

ds['train'] = train_df
ds['val'] = val_df

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=ds['train']['label'].values)
cw_heldout = torch.tensor(cw, dtype=torch.float).to(device)

train = DNAdataset(ds['train'], tokenizer)
val = DNAdataset(ds['val'], tokenizer)

test_OOD = DNAdataset(ds['test-heldout'], tokenizer)
test_ID = DNAdataset(ds['test-id'], tokenizer)
test_matched_ID = DNAdataset(ds['test-matched_id'], tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
for name, df in ds.items():
    total = len(df)
    vc = df['label'].value_counts().sort_index()
    print(f"{name:<20} n={total:>5}  class 0: {vc.get(0,0):>4} ({vc.get(0,0)/total*100:.1f}%)  class 1: {vc.get(1,0):>4} ({vc.get(1,0)/total*100:.1f}%)")

train                n=15692  class 0: 9554 (60.9%)  class 1: 6138 (39.1%)
test-heldout         n= 1569  class 0:  606 (38.6%)  class 1:  963 (61.4%)
test-id              n= 4905  class 0: 3004 (61.2%)  class 1: 1901 (38.8%)
test-matched_id      n= 1569  class 0:  606 (38.6%)  class 1:  963 (61.4%)
val                  n= 3924  class 0: 2459 (62.7%)  class 1: 1465 (37.3%)


## 2.2 lora

In [ ]:
# Lora training

#init model
lora_classifier = DNABERT2Classifier(pad_token=tokenizer.pad_token_id)

# train
lora_classifier = train_model(
    model = lora_classifier,
    tokenizer = tokenizer,
    train = train,
    val = val,
    lr=4e-5,
    batch_size=64,
    epochs=50,
    patience=5,
    class_weights=cw_heldout,
    save_path="saved_models/lora_classifier.pth"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


========================= Epoch: 0 =========================


Train loss: 0.6734948916648461


Val loss: 0.639049232006073
best val loss: 0.64 no imp epochs: 0
========================= Epoch: 1 =========================


Train loss: 0.5864591753579736


...

Val loss: 0.41743042560354354
best val loss: 0.39 no imp epochs: 3
========================= Epoch: 25 =========================


Train loss: 0.3093677675578652


Val loss: 0.4205103958806684
best val loss: 0.39 no imp epochs: 4
========================= Epoch: 26 =========================


Train loss: 0.30829301470421194


Val loss: 0.40917748861735864
best val loss: 0.39 no imp epochs: 5
Early stopping triggered.


In [ ]:
results = []

# evaluate
ys, ps, probs = predict(lora_classifier, test_ID, tokenizer)
results.append(record('DNABERT2_lora', 'ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(lora_classifier, test_matched_ID, tokenizer)
results.append(record('DNABERT2_lora', 'matched_ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(lora_classifier, test_OOD, tokenizer)
results.append(record('DNABERT2_lora', 'OOD', ys, ps, probs))

100%|██████████| 307/307 [00:08<00:00, 37.63it/s]


  DNABERT2_lora    ID         F1=0.8486  BalAcc=0.8399  AUC=0.9125


100%|██████████| 99/99 [00:02<00:00, 37.81it/s]


  DNABERT2_lora    matched_ID F1=0.9104  BalAcc=0.9243  AUC=0.9729


100%|██████████| 99/99 [00:02<00:00, 35.82it/s]

  DNABERT2_lora    OOD        F1=0.8468  BalAcc=0.8500  AUC=0.9274


## 2.3 no lora

In [ ]:
# train

#init model
classifier = DNABERT2Classifier(pad_token=tokenizer.pad_token_id)

# train
classifier = train_model(
    model = classifier,
    tokenizer = tokenizer,
    train = train,
    val = val,
    lr=4e-4,
    batch_size=64,
    epochs=50,
    patience=5,
    class_weights=cw_heldout,
    freeze_base=True,
    save_path="saved_models/classifier.pth"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


========================= Epoch: 0 =========================


Train loss: 0.6115642806621102


Val loss: 0.5320318259539143
best val loss: 0.53 no imp epochs: 0
========================= Epoch: 1 =========================


Train loss: 0.5147053649028143


...

Val loss: 0.4177477595306212
best val loss: 0.41 no imp epochs: 3
========================= Epoch: 27 =========================


Train loss: 0.390238793400245


Val loss: 0.42748385523596116
best val loss: 0.41 no imp epochs: 4
========================= Epoch: 28 =========================


Train loss: 0.3861118050488999


Val loss: 0.4174159837345923
best val loss: 0.41 no imp epochs: 5
Early stopping triggered.


In [ ]:
# evaluate
ys, ps, probs = predict(classifier, test_ID, tokenizer)
results.append(record('DNABERT2', 'ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(classifier, test_matched_ID, tokenizer)
results.append(record('DNABERT2', 'matched_ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(classifier, test_OOD, tokenizer)
results.append(record('DNABERT2', 'OOD', ys, ps, probs))

100%|██████████| 307/307 [00:08<00:00, 37.86it/s]


  DNABERT2         ID         F1=0.8210  BalAcc=0.8135  AUC=0.8904


100%|██████████| 99/99 [00:02<00:00, 37.49it/s]


  DNABERT2         matched_ID F1=0.8989  BalAcc=0.9103  AUC=0.9455


100%|██████████| 99/99 [00:02<00:00, 37.33it/s]

  DNABERT2         OOD        F1=0.8267  BalAcc=0.8311  AUC=0.9136


## 2.4 combined results

| Model | Split | F1 | Accuracy |
|---|---|---|---|
| DNABERT2 + LoRA | ID | 0.8486 | 0.8399 |
| DNABERT2 + LoRA | matched\_ID | 0.9104 | 0.9243 |
| DNABERT2 + LoRA | OOD | 0.8468 | 0.8500 |
| DNABERT2 (frozen) | ID | 0.8210 | 0.8135 |
| DNABERT2 (frozen) | matched\_ID | 0.8989 | 0.9103 |
| DNABERT2 (frozen) | OOD | 0.8267 | 0.8311 |

In [ ]:
print("Model Name       |    split   |  F1    |  acc")
print("-------------------------------------------------")
for result in results:
    print(f"{result['Model']:<16} | {result['Split']:<10} | {result['f1']:.4f} | {result['balanced_accuracy']:.4f}")

Model Name       |    split   |  F1    |  acc
--------------------------------------------------
DNABERT2_lora    | ID         | 0.8486 | 0.8399
DNABERT2_lora    | matched_ID | 0.9104 | 0.9243
DNABERT2_lora    | OOD        | 0.8468 | 0.8500
DNABERT2         | ID         | 0.8210 | 0.8135
DNABERT2         | matched_ID | 0.8989 | 0.9103
DNABERT2         | OOD        | 0.8267 | 0.8311


# 3 experimental result - dataset M

## 3.1 load data

In [ ]:
# imports
from datasets import load_dataset
from typing import Optional
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# load data
ds = pd.read_csv("dataset_m/promoter_final_dataset.csv")
ds = {n: ds[ds['split'] == n].reset_index(drop=True)
      for n in ['train', 'val', 'id_test', 'ood_test']}

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)


cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=ds['train']['label'].values)
cw_heldout = torch.tensor(cw, dtype=torch.float).to(device)

train = DNAdataset(ds['train'], tokenizer)
val = DNAdataset(ds['val'], tokenizer)
test_OOD = DNAdataset(ds['ood_test'], tokenizer)
test_ID = DNAdataset(ds['id_test'], tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 3.2 Lora

In [ ]:
# Lora training

#init model
lora_classifier = DNABERT2Classifier(pad_token=tokenizer.pad_token_id)

# train
lora_classifier = train_model(
    model = lora_classifier,
    tokenizer = tokenizer,
    train = train,
    val = val,
    lr=4e-5,
    batch_size=64,
    epochs=50,
    patience=5,
    class_weights=cw_heldout,
    save_path="saved_models/lora_classifier.pth"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


========================= Epoch: 0 =========================


Train loss: 0.6822688265683803


Val loss: 0.6406305781224879
best val loss: 0.64 no imp epochs: 0
========================= Epoch: 1 =========================


Train loss: 0.5853831081631335


...

Val loss: 0.4342117156924271
best val loss: 0.43 no imp epochs: 3
========================= Epoch: 20 =========================


Train loss: 0.37638407549325453


Val loss: 0.44410096472356375
best val loss: 0.43 no imp epochs: 4
========================= Epoch: 21 =========================


Train loss: 0.3706985082715116


Val loss: 0.44403333634864994
best val loss: 0.43 no imp epochs: 5
Early stopping triggered.


In [ ]:
results = []

# evaluate
ys, ps, probs = predict(lora_classifier, test_ID, tokenizer)
results.append(record('DNABERT2_lora', 'ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(lora_classifier, test_OOD, tokenizer)
results.append(record('DNABERT2_lora', 'OOD', ys, ps, probs))

100%|██████████| 162/162 [00:04<00:00, 33.58it/s]


  DNABERT2_lora    ID         F1=0.7678  BalAcc=0.8062  AUC=0.8840


100%|██████████| 233/233 [00:07<00:00, 33.22it/s]

  DNABERT2_lora    OOD        F1=0.6195  BalAcc=0.7471  AUC=0.9490


## 3.3 No lora

In [ ]:
# train

#init model
classifier = DNABERT2Classifier(pad_token=tokenizer.pad_token_id)

# train
classifier = train_model(
    model = classifier,
    tokenizer = tokenizer,
    train = train,
    val = val,
    lr=4e-4,
    batch_size=64,
    epochs=50,
    patience=5,
    class_weights=cw_heldout,
    freeze_base=True,
    save_path="saved_models/classifier.pth"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


========================= Epoch: 0 =========================


Train loss: 0.609677679361181


Val loss: 0.5378644044806318
best val loss: 0.54 no imp epochs: 0
========================= Epoch: 1 =========================


Train loss: 0.5029816296189389


...

Val loss: 0.5651750542768618
best val loss: 0.49 no imp epochs: 3
========================= Epoch: 6 =========================


Train loss: 0.47732240405488524


Val loss: 0.4935012692358436
best val loss: 0.49 no imp epochs: 4
========================= Epoch: 7 =========================


Train loss: 0.4753692441798271


Val loss: 0.49094231463060145
best val loss: 0.49 no imp epochs: 5
Early stopping triggered.


In [ ]:
# evaluate
ys, ps, probs = predict(classifier, test_ID, tokenizer)
results.append(record('DNABERT2', 'ID', ys, ps, probs))

# evaluate
ys, ps, probs = predict(classifier, test_OOD, tokenizer)
results.append(record('DNABERT2', 'OOD', ys, ps, probs))

100%|██████████| 162/162 [00:04<00:00, 34.20it/s]


  DNABERT2         ID         F1=0.7440  BalAcc=0.7595  AUC=0.8517


100%|██████████| 233/233 [00:07<00:00, 33.03it/s]

  DNABERT2         OOD        F1=0.5890  BalAcc=0.7230  AUC=0.9315


## 3.4 print results



| Model | Split | F1 | Accuracy |
|---|---|---|---|
| DNABERT2 + LoRA | ID | 0.7678 | 0.8062 |
| DNABERT2 + LoRA | OOD | 0.6195 | 0.7471 |
| DNABERT2 (frozen) | ID | 0.7440 | 0.7595 |
| DNABERT2 (frozen) | OOD | 0.5890 | 0.7230 |

In [ ]:
print("Model Name       |    split   |  F1    |  acc")
print("-------------------------------------------------")
for result in results:
    print(f"{result['Model']:<16} | {result['Split']:<10} | {result['f1']:.4f} | {result['balanced_accuracy']:.4f}")

Model Name       |    split   |  F1    |  acc
-------------------------------------------------
DNABERT2_lora    | ID         | 0.7678 | 0.8062
DNABERT2_lora    | OOD        | 0.6195 | 0.7471
DNABERT2         | ID         | 0.7440 | 0.7595
DNABERT2         | OOD        | 0.5890 | 0.7230
